# 02 · Filtering, features, and FFT

Here we take one of UCI HAR's short inertial-signal segments and see how smoothing, summary statistics, and a frequency spectrum change the picture. The activity labels are used only to compare examples; this notebook does not train a classifier.

## A. Load and validate both original splits

This notebook runs on its own. It loads all nine inertial signal files per split and checks labels, participant IDs, and 128-sample rows. It uses the original train/test folders; it never joins windows into a pretend continuous timeline.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
FS, N = 50, 128
KINDS, AXES = ("total_acc", "body_acc", "body_gyro"), ("x", "y", "z")
def project_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p/"notebooks"/"01_signal_exploration.ipynb").is_file(): return p
    raise FileNotFoundError("Open Jupyter from wearable-signal-lab/ or its notebooks/ folder.")
ROOT=project_root()
choices=[p/"data"/"raw"/"UCI HAR Dataset" for p in (ROOT,*ROOT.parents)]
DATA=next((p for p in choices if p.is_dir()),None)
if DATA is None:
    raise FileNotFoundError("UCI HAR data missing. Download from https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones, extract 'UCI HAR Dataset' to data/raw/ or reuse neighboring wearable-ai-101/data/raw/. See data/README.md. No fake data is made.")
a=pd.read_csv(DATA/"activity_labels.txt",sep=r"\s+",header=None,names=["id","name"])
ACT=dict(zip(a.id,a.name))
def load(split):
    d=DATA/split; sigdir=d/"Inertial Signals"
    paths=[d/f"y_{split}.txt",d/f"subject_{split}.txt"]+[sigdir/f"{kind}_{axis}_{split}.txt" for kind in KINDS for axis in AXES]
    missing=[str(p.relative_to(DATA)) for p in paths if not p.is_file()]
    if missing: raise FileNotFoundError(f"Missing {split} files: {missing}. Check data/README.md.")
    y=np.loadtxt(d/f"y_{split}.txt",dtype=int).reshape(-1); sub=np.loadtxt(d/f"subject_{split}.txt",dtype=int).reshape(-1)
    signals={}
    for kind in KINDS:
        signals[kind]={}
        for axis in AXES:
            p=sigdir/f"{kind}_{axis}_{split}.txt"
            v=pd.read_csv(p,sep=r"\s+",header=None).to_numpy(dtype=np.float32)
            if v.shape!=(len(y),N): raise ValueError(f"{p.name} has {v.shape}; expected ({len(y)}, {N}).")
            signals[kind][axis]=v
    if len(sub)!=len(y): raise ValueError(f"{split} subject IDs and labels are misaligned.")
    if set(y)-set(ACT): raise ValueError(f"Unknown activity IDs: {set(y)-set(ACT)}")
    return {"y":y,"subjects":sub,"signals":signals}
train,test=load("train"),load("test")
for name,data,expected in (("train",train,7352),("test",test,2947)):
    if len(data["y"])!=expected: raise ValueError(f"Expected {expected} {name} windows, got {len(data['y'])}.")
print(f"Data: {DATA}; train={len(train['y'])}, test={len(test['y'])}; each row={N} samples ({N/FS:.2f}s) at {FS} Hz")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | Matplotlib {plt.matplotlib.__version__} | SciPy {scipy.__version__}")

## B. Choose a real example

Use the first WALKING training window and its body-acceleration X and gyroscope X axes. One axis makes filter behavior clear; later, X/Y/Z are combined into a 3D magnitude.

In [ ]:
idx=int(np.flatnonzero(train["y"]==1)[0]); name=ACT[int(train["y"][idx])]
t=np.arange(N)/FS
acc=train["signals"]["body_acc"]["x"][idx].astype(float)
gyro=train["signals"]["body_gyro"]["x"][idx].astype(float)
print(f"Window {idx}: {name}; 128 samples span 2.56 s. UCI signals are already preprocessed.")

## C. Moving-average smoothing

A moving average replaces a point with the mean of nearby values. Five samples span 0.10 s at 50 Hz. We use a centered average for offline plots: it uses both past and future values, so it is not a real-time filter. Wider averages smooth more but can hide real peaks. Edge padding avoids artificial zero dips at the ends.

In [ ]:
def moving_average(x,width):
    if width<1 or width%2==0: raise ValueError("Use a positive odd width.")
    return np.convolve(np.pad(x,(width//2,width//2),mode="edge"),np.ones(width)/width,mode="valid")
WIDTH=5
acc_ma=moving_average(acc,WIDTH);gyro_ma=moving_average(gyro,WIDTH)
fig,axs=plt.subplots(2,1,figsize=(10,6),sharex=True)
for ax,raw,smooth,unit,title in ((axs[0],acc,acc_ma,"g","Body acceleration X"),(axs[1],gyro,gyro_ma,"rad/s","Body gyroscope X")):
    ax.plot(t,raw,label="Provided signal",alpha=.55);ax.plot(t,smooth,label=f"Moving average ({WIDTH} samples)")
    ax.set_ylabel(unit);ax.set_title(title);ax.grid(alpha=.25);ax.legend()
axs[-1].set_xlabel("Time (s)");plt.tight_layout();plt.show()
print(f"A {WIDTH}-sample average spans {WIDTH/FS:.2f} s.")

## D. Butterworth low-pass filter

A low-pass filter reduces faster changes while retaining slower ones. Nyquist is half the sample rate, 25 Hz here; cutoff must be lower. We use a 4th-order, 5 Hz Butterworth filter only as an example, not as a universal best setting. `sosfiltfilt` filters forward and backward, avoiding net phase delay offline, but uses future samples, is not causal for live use, and can distort edges. Extra filtering can remove meaningful motion too.

In [ ]:
from scipy.signal import butter,sosfiltfilt,sosfreqz
ORDER,CUTOFF=4,5.0
NYQUIST=FS/2
if not 0<CUTOFF<NYQUIST:raise ValueError("Cutoff must be between 0 and Nyquist.")
sos=butter(ORDER,CUTOFF,btype="lowpass",fs=FS,output="sos")
acc_lp=sosfiltfilt(sos,acc);gyro_lp=sosfiltfilt(sos,gyro)
fig,axs=plt.subplots(2,1,figsize=(10,6),sharex=True)
for ax,raw,ma,lp,unit,title in ((axs[0],acc,acc_ma,acc_lp,"g","Body acceleration X"),(axs[1],gyro,gyro_ma,gyro_lp,"rad/s","Body gyroscope X")):
    ax.plot(t,raw,label="Provided",alpha=.4);ax.plot(t,ma,label="Moving average");ax.plot(t,lp,label="Butterworth 5 Hz")
    ax.set_ylabel(unit);ax.set_title(title);ax.grid(alpha=.25);ax.legend()
axs[-1].set_xlabel("Time (s)");plt.tight_layout();plt.show()
freq,h=sosfreqz(sos,worN=512,fs=FS)
plt.figure(figsize=(8,3.5));plt.plot(freq,20*np.log10(np.maximum(abs(h),1e-8)))
plt.axvline(CUTOFF,color="red",ls="--",label="5 Hz cutoff");plt.axvline(NYQUIST,color="gray",ls=":",label="25 Hz Nyquist")
plt.xlim(0,NYQUIST);plt.ylim(-80,5);plt.xlabel("Frequency (Hz)");plt.ylabel("Gain (dB)");plt.title("Butterworth frequency response");plt.grid(alpha=.25);plt.legend();plt.show()

## E. Window statistics and signal magnitude

For every original 128-sample window, calculate mean, population standard deviation, minimum, maximum, and RMS for each axis. Calculate magnitude at each time as sqrt(X² + Y² + Z²), then summarize it with the same statistics. Magnitude combines axis size but discards direction. These are descriptive features, not predictions.

In [ ]:
def summarize(matrix,prefix):
    return {f"{prefix}_{stat}":fn(matrix,axis=1) for stat,fn in
            (("mean",np.mean),("std",np.std),("min",np.min),("max",np.max))} |            {f"{prefix}_rms":np.sqrt(np.mean(matrix**2,axis=1))}
frames=[]
for kind in KINDS:
    cols={}
    for axis in AXES:cols.update(summarize(train["signals"][kind][axis],f"{kind}_{axis}"))
    xyz=np.stack([train["signals"][kind][axis] for axis in AXES],axis=2)
    magnitude=np.sqrt(np.sum(xyz**2,axis=2));cols.update(summarize(magnitude,f"{kind}_magnitude"))
    frames.append(pd.DataFrame(cols))
features=pd.concat(frames,axis=1);features["activity_id"]=train["y"];features["activity"]=features.activity_id.map(ACT)
display(features.head().round(4))
print("Each row summarizes one supplied sensor window; numeric features:",features.select_dtypes(include="number").shape[1])

## F. Compare characteristics across activities

Compare body-acceleration magnitude RMS, total-acceleration magnitude mean, and gyroscope magnitude RMS. The table reports mean and standard deviation per activity; the box plot shows window-to-window spread. These are everyday smartphone activity comparisons only.

In [ ]:
measures=["body_acc_magnitude_rms","total_acc_magnitude_mean","body_gyro_magnitude_rms"]
display(features.groupby("activity",sort=False)[measures].agg(["mean","std"]).round(4))
order=[ACT[i] for i in sorted(ACT)]
boxes=[features.loc[features.activity==a,"body_acc_magnitude_rms"].to_numpy() for a in order]
fig,ax=plt.subplots(figsize=(10,4.5));ax.boxplot(boxes,tick_labels=order,showfliers=False)
ax.set(title="Body-acceleration magnitude RMS by activity",xlabel="Activity",ylabel="RMS magnitude (g)")
plt.xticks(rotation=25,ha="right");ax.grid(axis="y",alpha=.25);plt.tight_layout();plt.show()
print("Overlapping box ranges mean this one statistic does not cleanly separate every window.")

## G. FFT: frequency-domain representation

The FFT represents a finite signal as frequency components. We subtract the average, apply a Hann taper to reduce edge leakage, and plot one-sided amplitude. With N samples, bin spacing is sample rate / N. A longer window improves frequency spacing but hides local changes; a peak does not explain its cause.

In [ ]:
from scipy.fft import rfft,rfftfreq
def spectrum(x,fs):
    n=len(x);taper=np.hanning(n);z=rfft((x-np.mean(x))*taper);amp=np.abs(z)/taper.sum()
    if n%2==0:amp[1:-1]*=2
    else:amp[1:]*=2
    return rfftfreq(n,d=1/fs),amp
fig,axs=plt.subplots(2,1,figsize=(9,6),sharex=True)
for ax,x,title,unit in ((axs[0],acc,"Body acceleration X","g"),(axs[1],gyro,"Body gyroscope X","rad/s")):
    f,a=spectrum(x,FS);ax.plot(f,a);ax.set_xlim(0,FS/2);ax.set_ylabel(f"Amplitude ({unit})");ax.set_title(title);ax.grid(alpha=.25)
axs[-1].set_xlabel("Frequency (Hz)");plt.tight_layout();plt.show()
print(f"128 samples give {FS/128:.3f} Hz FFT-bin spacing.")

## H. Window size changes both features and spectra

Subdivide this same 128-sample row into non-overlapping chunks of 32, 64, and 128 samples. Short chunks give more local values but coarser frequency bins. Long chunks give finer bin spacing but fewer summaries. These are pieces of an existing dataset window, not new recordings.

In [ ]:
sizes=(32,64,128);records=[];rms_by_size={}
fig,axs=plt.subplots(3,1,figsize=(10,7),sharex=True)
for ax,size in zip(axs,sizes):
    rms=[]
    for chunk_id,start in enumerate(range(0,N,size)):
        chunk=acc[start:start+size];rms.append(float(np.sqrt(np.mean(chunk**2))))
        records.append({"samples":size,"duration_s":size/FS,"fft_bin_hz":FS/size,"chunk":chunk_id,
                        "mean_g":float(chunk.mean()),"std_g":float(chunk.std()),"rms_g":rms[-1]})
    rms_by_size[size]=rms;ax.plot(t,acc)
    for edge in range(size,N,size):ax.axvline(edge/FS,color="red",ls="--",alpha=.6)
    ax.set_title(f"{size} samples = {size/FS:.2f} s");ax.set_ylabel("Body accel X (g)");ax.grid(alpha=.2)
axs[-1].set_xlabel("Time in original row (s)");plt.tight_layout();plt.show()
window_table=pd.DataFrame(records);display(window_table.round(4))
fig,ax=plt.subplots(figsize=(8,3.5))
for size,rms in rms_by_size.items():ax.plot(range(1,len(rms)+1),rms,marker="o",label=f"N={size}, bins={FS/size:.2f} Hz")
ax.set(title="RMS by chunk",xlabel="Chunk number",ylabel="RMS (g)");ax.grid(alpha=.25);ax.legend();plt.show()
fig,ax=plt.subplots(figsize=(8,3.5))
for size in sizes:
    f,a=spectrum(acc[:size],FS);ax.plot(f,a,marker=".",label=f"N={size}: {FS/size:.2f} Hz bins")
ax.set(xlim=(0,FS/2),xlabel="Frequency (Hz)",ylabel="Amplitude (g)",title="FFT spacing changes with window size");ax.grid(alpha=.25);ax.legend();plt.show()
print("Each chunk comes from one original UCI segment.")

## I. Glossary and independent exercises

- **Moving average:** smooths by averaging neighbors.
- **Butterworth:** low-pass design with a smooth passband.
- **Cutoff:** approximate start of stronger attenuation (about −3 dB for Butterworth).
- **Nyquist:** half the sampling rate; faster frequencies cannot be uniquely represented.
- **Causal filter:** uses present/past samples; useful live, may add delay.
- **Forward/backward filtering:** offline, zero net phase shift, uses future samples and may affect edges.
- **RMS:** square root of mean squared amplitude.
- **Magnitude:** sqrt(X²+Y²+Z²); combines axes but loses direction.
- **FFT:** finite-signal frequency representation.
- **Hann taper:** softens edges before FFT; reduces leakage but broadens peaks.
- **Frequency resolution:** sample rate divided by sample count.
- **Spectral leakage:** energy spread across bins by finite segment edges.
- **Distortion:** unwanted change to shape, timing, or amplitude.

**Exercises:** (1) Try moving-average widths 3, 9, 15; compare RMS. (2) Compare Butterworth cutoffs 2, 5, 10 Hz in plots and response. (3) Compare WALKING and LAYING magnitude summaries. (4) Recompute one magnitude from the three axes. (5) Explain causal vs forward/backward filtering. (6) Predict FFT spacing for 32 and 128 samples, then check the plots. Do not interpret these activity results as fatigue or driver strain.